In [1]:
import os,sys
sys.path.insert(1, os.path.join(os.getcwd()  , '..'))

In [2]:
import shallowsim as sb
import pandas as pd
import math

In [ ]:
args = sb.ModelArgs()
gpu_blackwell = sb.get_gpu_info('./device/gpu_info.csv', print_console=False)
print(f"成功加载 {len(gpu_blackwell)} 种GPU配置:", list(gpu_blackwell.keys())) 

成功加载 11 种GPU配置: ['H200', 'H800', 'H20', 'H20-3E', 'MI300X', 'MI308X', 'TG260', 'TG260X-32', 'TG260X-64', 'TG260X-128', 'TG260X-288']


In [ ]:
# 新的参数配置
seq_len_list = [4383, 8766, 17532, 35064, 70128]  # 序列长度列表
kv_cache_rate = 0.563                              # KV缓存命中率
tp_list = [1, 2, 4]                               # Tensor Parallel 配置
dp_list = [8, 16, 32, 64, 128]                    # Data Parallel 配置

# 原有参数保留备用
seq_len = 4383  # 默认序列长度
decode_len = 1210
bs_list = [16, 32, 64, 128, 256, 512]
eplist = [8, 16, 36, 72, 144, 320]

In [5]:
# 批量测试不同参数组合
import itertools
import numpy as np

# 创建结果存储
results_df = []

print("开始批量测试prefill性能...")
print(f"序列长度: {seq_len_list}")
print(f"TP配置: {tp_list}")
print(f"DP配置: {dp_list}")
print(f"KV缓存命中率: {kv_cache_rate}")
print("="*50)

# 遍历所有参数组合
for seq_len, tp, dp in itertools.product(seq_len_list, tp_list, dp_list):
    print(f"测试: seq_len={seq_len}, tp={tp}, dp={dp}")
    
    try:
        # 计算prefill时间
        detail, summary = sb.prefill_time(args, gpu_blackwell, seq_len, kv_cache_rate, tp=tp, dp=dp, print_console=False)
        
        # 计算throughput (tokens/second/device)
        throughput_per_device = summary.apply(lambda x: seq_len/tp * (1000 / x)).loc['Sum']
        
        # 保存结果
        for gpu_name in gpu_blackwell.keys():
            if gpu_name in throughput_per_device.index:
                results_df.append({
                    'GPU': gpu_name,
                    'seq_len': seq_len,
                    'tp': tp,
                    'dp': dp,
                    'kv_cache_rate': kv_cache_rate,
                    'prefill_time_ms': summary.loc['Sum', gpu_name],
                    'throughput_tokens_per_sec_per_device': throughput_per_device[gpu_name]
                })
        
    except Exception as e:
        print(f"  错误: {e}")
        continue

# 转换为DataFrame
results_df = pd.DataFrame(results_df)
print(f"\n完成测试，共收集 {len(results_df)} 条结果")


开始批量测试prefill性能...
序列长度: [4383, 8766, 17532, 35064, 70128]
TP配置: [1, 2, 4]
DP配置: [8, 16, 32, 64, 128]
KV缓存命中率: 0.563
测试: seq_len=4383, tp=1, dp=8
测试: seq_len=4383, tp=1, dp=16
测试: seq_len=4383, tp=1, dp=32
测试: seq_len=4383, tp=1, dp=64
测试: seq_len=4383, tp=1, dp=128
测试: seq_len=4383, tp=2, dp=8
测试: seq_len=4383, tp=2, dp=16
测试: seq_len=4383, tp=2, dp=32
测试: seq_len=4383, tp=2, dp=64
测试: seq_len=4383, tp=2, dp=128
测试: seq_len=4383, tp=4, dp=8
测试: seq_len=4383, tp=4, dp=16
测试: seq_len=4383, tp=4, dp=32
测试: seq_len=4383, tp=4, dp=64
测试: seq_len=4383, tp=4, dp=128
测试: seq_len=8766, tp=1, dp=8
测试: seq_len=8766, tp=1, dp=16
测试: seq_len=8766, tp=1, dp=32
测试: seq_len=8766, tp=1, dp=64
测试: seq_len=8766, tp=1, dp=128
测试: seq_len=8766, tp=2, dp=8
测试: seq_len=8766, tp=2, dp=16
测试: seq_len=8766, tp=2, dp=32
测试: seq_len=8766, tp=2, dp=64
测试: seq_len=8766, tp=2, dp=128
测试: seq_len=8766, tp=4, dp=8
测试: seq_len=8766, tp=4, dp=16
测试: seq_len=8766, tp=4, dp=32
测试: seq_len=8766, tp=4, dp=64
测试: seq_len=87

In [ ]:
# 结果分析和可视化
print("=== 性能分析结果 ===\n")

# 1. 按GPU类型分组，显示不同配置下的性能
print("1. 不同GPU的最佳性能配置:")
best_config_per_gpu = results_df.loc[results_df.groupby('GPU')['throughput_tokens_per_sec_per_device'].idxmax()]
print(best_config_per_gpu[['GPU', 'seq_len', 'tp', 'dp', 'throughput_tokens_per_sec_per_device']].to_string(index=False))

print("\n" + "="*80)

# 2. 按序列长度分析
print("2. 不同序列长度下的平均性能 (tokens/sec/device):")
seq_len_analysis = results_df.groupby(['seq_len'])['throughput_tokens_per_sec_per_device'].agg(['mean', 'std', 'max']).round(1)
print(seq_len_analysis)

print("\n" + "="*80)

# 3. 按TP配置分析
print("3. 不同TP配置下的平均性能:")
tp_analysis = results_df.groupby(['tp'])['throughput_tokens_per_sec_per_device'].agg(['mean', 'std', 'max']).round(1)
print(tp_analysis)

print("\n" + "="*80)

# 4. 按DP配置分析
print("4. 不同DP配置下的平均性能:")
dp_analysis = results_df.groupby(['dp'])['throughput_tokens_per_sec_per_device'].agg(['mean', 'std', 'max']).round(1)
print(dp_analysis)




=== 性能分析结果 ===

1. 不同GPU的最佳性能配置:
       GPU  seq_len  tp  dp  throughput_tokens_per_sec_per_device
       H20     4383   1 128                           1337.160865
    H20-3E     4383   1 128                           1337.547454
      H200     4383   1 128                           8590.487418
      H800     4383   1 128                           8574.565854
    MI300X     4383   1 128                          11999.548610
    MI308X     4383   1 128                           1774.235263
     TG260     4383   1 128                           2961.131940
TG260X-128     4383   1 128                           2961.131940
TG260X-288     4383   1 128                           2961.131940
 TG260X-32     4383   1 128                           2961.131940
 TG260X-64     4383   1 128                           2961.131940

2. 不同序列长度下的平均性能 (tokens/sec/device):
           mean     std      max
seq_len                         
4383     3927.7  3027.9  11999.5
8766     2926.6  2279.4   8790.1
17532

In [ ]:
# 创建专门对比单个GPU在不同sequence length下性能的函数
def prefill_time_by_seq_len(args, gpu, seq_len_list, kv_cache_rate, tp, dp, print_console=False):
    """
    对比单个GPU在不同sequence length下的prefill性能
    
    Args:
        args: ModelArgs对象
        gpu: 单个GPU对象
        seq_len_list: 序列长度列表 
        kv_cache_rate: KV缓存命中率
        tp: Tensor Parallel配置
        dp: Data Parallel配置
        print_console: 是否打印到控制台
    
    Returns:
        df: 详细性能数据，行为算子，列为不同序列长度
        df2: 汇总性能数据(计算时间、通信时间、总时间)
    """
    import pandas as pd
    
    # 创建列名：包含GPU名称和所有序列长度
    columns = ['GPU'] + [f'seq_{seq_len}' for seq_len in seq_len_list] 
    df = pd.DataFrame(columns=columns)
    df2 = pd.DataFrame(columns=columns)
    
    n_sparse_layers = args.n_layers - args.n_dense_layers
    
    # 添加Layers行(说明每个算子运行多少层)
    layers_row = [f'{gpu.gpu_type}_Layers'] + [f'{args.n_dense_layers}' if i < 2 else f'{n_sparse_layers}' 
                                               for i in range(len(seq_len_list))]
    
    # 为每个算子类型创建行
    operator_names = ['MLA', 'DenseMLP', 'TP_MLA', 'Shared Expert', 'Combine', 'Overlap1', 
                     'Routed Expert', 'Dispatch', 'Overlap2']
    
    # 计算每个序列长度下的性能
    results = {}
    for seq_len in seq_len_list:
        # 调用_prefill_time获取单个配置的结果
        dense_mla, dense_mlp, tp_mla, shared, combine, routed, dispatch = sb._prefill_time(
            args, gpu, seq_len, kv_cache_rate, tp, dp)
        
        overlap1 = combine - (tp_mla + shared)
        overlap2 = dispatch - routed
        
        results[seq_len] = {
            'MLA': dense_mla,
            'DenseMLP': dense_mlp, 
            'TP_MLA': tp_mla,
            'Shared Expert': shared,
            'Combine': combine,
            'Overlap1': overlap1,
            'Routed Expert': routed,
            'Dispatch': dispatch,
            'Overlap2': overlap2
        }
        
        # 计算汇总数据
        comp_time = args.n_dense_layers * (dense_mla + dense_mlp) + n_sparse_layers * (tp_mla + shared + routed)
        comm_time = n_sparse_layers * (combine + dispatch)
        sum_time = comp_time
        if overlap1 > 0:
            sum_time += overlap1 * n_sparse_layers
        if overlap2 > 0:
            sum_time += overlap2 * n_sparse_layers
            
        results[seq_len]['Compute'] = comp_time
        results[seq_len]['Comm'] = comm_time
        results[seq_len]['Sum'] = sum_time
    
    # 构建详细性能DataFrame
    for op_name in operator_names:
        row_data = [op_name] + [results[seq_len][op_name] for seq_len in seq_len_list]
        df.loc[len(df)] = row_data
    
    # 构建汇总性能DataFrame  
    for summary_name in ['Compute', 'Comm', 'Sum']:
        row_data = [summary_name] + [results[seq_len][summary_name] for seq_len in seq_len_list]
        df2.loc[len(df2)] = row_data
    
    # 设置索引并转置
    df = df.set_index('GPU').T
    df2 = df2.set_index('GPU').T
    
    if print_console:
        print(f"=== {gpu.gpu_type} GPU 在不同序列长度下的Prefill性能对比 ===")
        print(f"配置: TP={tp}, DP={dp}, KV缓存命中率={kv_cache_rate}")
        print()
        print("详细算子性能 (毫秒):")
        print(df.to_markdown(floatfmt=".3f"))
        print()
        print("性能汇总 (毫秒):")
        print(df2.to_markdown(floatfmt=".3f"))
        print()
        
        # 计算吞吐量
        throughput_df = pd.DataFrame(index=df2.index, columns=df2.columns)
        for seq_len in seq_len_list:
            col_name = f'seq_{seq_len}'
            if col_name in df2.columns:
                total_time_ms = df2.loc['Sum', col_name] 
                throughput_per_device = (seq_len / tp) * (1000 / total_time_ms)  # tokens/sec/device
                throughput_df.loc['Throughput_tokens_per_sec_per_device', col_name] = throughput_per_device
        
        print("吞吐量 (tokens/sec/device):")
        print(throughput_df.dropna().to_markdown(floatfmt=".1f"))
    
    return df, df2


In [ ]:
# 测试TG260 GPU在不同序列长度下的性能对比
print("=== TG260 GPU在不同序列长度下的Prefill性能对比 ===\n")

# 选择要对比的序列长度
seq_len_list = [4383, 8766, 17532, 35064, 70128]
kv_cache_rate = 0.563
tp = 4  # Tensor Parallel
dp = 8  # Data Parallel

# 获取TG260 GPU
tg260_gpu = gpu_blackwell['TG260']

# 调用我们的新函数
detail_tg260, summary_tg260 = prefill_time_by_seq_len(
    args, 
    tg260_gpu, 
    seq_len_list, 
    kv_cache_rate, 
    tp, 
    dp, 
    print_console=True
)


In [ ]:
# 访问详细数据和进一步分析
print("=== 数据访问示例 ===\n")

print("1. 详细算子性能数据 (detail_tg260):")
print(detail_tg260)
print()

print("2. 性能汇总数据 (summary_tg260):")  
print(summary_tg260)
print()

print("3. 性能缩放分析:")
# 计算序列长度翻倍时的性能缩放
seq_lens = [4383, 8766, 17532, 35064, 70128]
print("序列长度翻倍的性能缩放效率:")
for i in range(1, len(seq_lens)):
    current_seq = seq_lens[i]
    prev_seq = seq_lens[i-1]
    
    current_time = summary_tg260.loc['Sum', f'seq_{current_seq}']
    prev_time = summary_tg260.loc['Sum', f'seq_{prev_seq}']
    
    scaling_ratio = current_time / prev_time
    theoretical_ratio = current_seq / prev_seq
    efficiency = theoretical_ratio / scaling_ratio
    
    print(f"  {prev_seq} -> {current_seq}: 实际={scaling_ratio:.2f}x, 理论={theoretical_ratio:.2f}x, 效率={efficiency:.1%}")

print()
print("4. 不同配置对比:")
print("你可以尝试不同的TP/DP配置:")
print("例如: prefill_time_by_seq_len(args, tg260_gpu, [4383, 17532], 0.563, tp=1, dp=16)")
print("     prefill_time_by_seq_len(args, tg260_gpu, [4383, 17532], 0.563, tp=2, dp=32)")


In [ ]:
# 创建一个更简化的版本，类似原始prefill_time的调用方式
def prefill_time_seq_comparison(args, gpu_name, seq_len_list, kv_cache_rate, tp, dp):
    """
    简化版本：对比指定GPU在不同序列长度下的性能
    返回格式类似原始prefill_time函数
    """
    # 创建只包含指定GPU的字典
    gpu_dict = {gpu_name: gpu_blackwell[gpu_name]}
    
    # 创建结果DataFrame，列为不同序列长度
    columns = ['GPU', 'Layers'] + [f'seq_{seq_len}' for seq_len in seq_len_list]
    detail = pd.DataFrame(columns=columns)
    summary = pd.DataFrame(columns=columns)
    
    n_sparse_layers = args.n_layers - args.n_dense_layers
    
    # 添加Layers行
    detail.loc[len(detail)] = ['Layers', args.n_dense_layers] + [args.n_dense_layers if i < 2 else n_sparse_layers 
                                                                 for i in range(len(seq_len_list))]
    
    # 算子名称
    operator_names = ['MLA', 'DenseMLP', 'TP_MLA', 'Shared Expert', 'Combine', 'Overlap1', 
                     'Routed Expert', 'Dispatch', 'Overlap2']
    
    # 为每个序列长度计算性能
    results = {}
    for seq_len in seq_len_list:
        detail_single, summary_single = sb.prefill_time(args, gpu_dict, seq_len, kv_cache_rate, tp, dp, print_console=False)
        results[seq_len] = {
            'detail': detail_single[gpu_name],  # 获取该GPU的详细数据
            'summary': summary_single[gpu_name]  # 获取该GPU的汇总数据
        }
    
    # 构建详细性能表
    for op_name in operator_names:
        row = [op_name, ''] + [results[seq_len]['detail'][op_name] for seq_len in seq_len_list]
        detail.loc[len(detail)] = row
    
    # 构建汇总表
    for summary_name in ['Compute', 'Comm', 'Sum']:
        row = [summary_name, ''] + [results[seq_len]['summary'][summary_name] for seq_len in seq_len_list]
        summary.loc[len(summary)] = row
    
    # 设置索引并转置，保持与原始格式一致
    detail = detail.set_index('GPU').T
    summary = summary.set_index('GPU').T
    
    return detail, summary

# 测试新的简化函数
print("=== 使用简化函数进行TG260性能对比 ===")
seq_lens_short = [4383, 17532, 70128]  # 选择3个代表性长度进行快速对比

detail_simple, summary_simple = prefill_time_seq_comparison(
    args, 'TG260', seq_lens_short, 0.563, tp=4, dp=8
)

print("详细性能数据:")
print(detail_simple)
print()
print("汇总性能数据:")
print(summary_simple)


In [7]:
# 创建性能对比表格
print("=== 详细性能对比表格 ===\n")

# 创建透视表 - 按GPU和序列长度分组，显示最佳TP/DP配置下的性能
pivot_table = results_df.loc[results_df.groupby(['GPU', 'seq_len'])['throughput_tokens_per_sec_per_device'].idxmax()]
performance_matrix = pivot_table.pivot_table(
    index='GPU', 
    columns='seq_len', 
    values='throughput_tokens_per_sec_per_device',
    aggfunc='first'
).round(1)

print("不同GPU在各序列长度下的最佳性能 (tokens/sec/device):")
print(performance_matrix)

print("\n" + "="*80)

# 显示对应的最佳配置
print("对应的最佳TP/DP配置:")
config_matrix = pivot_table.copy()
config_matrix['config'] = config_matrix.apply(lambda x: f"TP{x['tp']}/DP{x['dp']}", axis=1)
config_table = config_matrix.pivot_table(
    index='GPU',
    columns='seq_len', 
    values='config',
    aggfunc='first'
)
print(config_table)

print("\n" + "="*80)

# 性能缩放效率分析
print("序列长度扩展的性能缩放效率:")
baseline_seq = 4383
scaling_efficiency = performance_matrix.copy()
for col in scaling_efficiency.columns:
    if col != baseline_seq:
        scaling_efficiency[col] = scaling_efficiency[col] / scaling_efficiency[baseline_seq] / (col / baseline_seq)
        
scaling_efficiency = scaling_efficiency.drop(columns=[baseline_seq]).round(3)
print("(相对于4383序列长度的理论线性缩放效率，1.0为完美缩放)")
print(scaling_efficiency)


=== 详细性能对比表格 ===

不同GPU在各序列长度下的最佳性能 (tokens/sec/device):
seq_len       4383    8766    17532   35064   70128
GPU                                                
H20          1337.2   968.8   624.1   364.5   199.0
H20-3E       1337.5   968.9   624.1   364.5   199.0
H200         8590.5  6273.4  4053.5  2370.0  1294.0
H800         8574.6  6269.1  4052.6  2369.8  1294.0
MI300X      11999.5  8790.1  5686.7  3326.3  1816.5
MI308X       1774.2  1285.8   828.3   483.8   264.1
TG260        2961.1  2156.3  1391.8   813.5   444.1
TG260X-128   2961.1  2156.3  1391.8   813.5   444.1
TG260X-288   2961.1  2156.3  1391.8   813.5   444.1
TG260X-32    2961.1  2156.3  1391.8   813.5   444.1
TG260X-64    2961.1  2156.3  1391.8   813.5   444.1

对应的最佳TP/DP配置:
seq_len         4383       8766       17532      35064      70128
GPU                                                              
H20         TP1/DP128  TP1/DP128  TP1/DP128  TP1/DP128  TP1/DP128
H20-3E      TP1/DP128  TP1/DP128  TP1/DP128  TP1/DP128

In [ ]:
detail,summary = sb.prefill_time(args,gpu_blackwell,seq_len, kv_cache_rate, tp=4, dp=8)

In [ ]:
detail

In [ ]:
summary

In [ ]:
tp=4
_ , ttft_sum = sb.prefill_time(args,gpu_blackwell,seq_len, kv_cache_rate, tp=tp, dp=8, print_console=False)
print(ttft_sum.apply(lambda x: seq_len/tp * (1000/ x)).loc['Sum'].to_markdown(floatfmt=".1f"))


| GPU        |    Sum |
|:-----------|-------:|
| H200       | 1116.6 |
| H800       | 1103.8 |
| H20        |  173.1 |
| H20-3E     |  173.1 |
| MI300X     | 1561.3 |
| MI308X     |  229.7 |
| TG260      |  383.8 |
| TG260X-32  |  383.8 |
| TG260X-64  |  383.8 |
| TG260X-128 |  383.8 |
| TG260X-288 |  383.8 |
